## Load data

In [1]:
import pandas as pd
import numpy as np

#Load clean transaction data and targets (clean data)
df_train = pd.read_csv("data/train_clean.csv", parse_dates=["order_date","pack_date"])
df_valid = pd.read_csv("data/valid_clean.csv", parse_dates=["order_date","pack_date"])

df_train_targets = pd.read_csv("data/train_targets.csv")
df_valid_targets = pd.read_csv("data/valid_targets.csv")

print("Train transactions:",df_train.shape)
print("Valid transactions:",df_valid.shape)

Train transactions: (219128, 27)
Valid transactions: (55183, 27)


In [2]:
#Reference date: last day of the transaction period
#We use this to calculate recency (how many days since last purchase)
reference_date = pd.Timestamp("2017-12-31")
reference_date

Timestamp('2017-12-31 00:00:00')

## RFM + pruschase behavior features

In [52]:
def compute_rfm_features(df):
    """
    Computes RFM and purchase behavior features, aggregated at customer level.
    Input: transaction-level dataframe (one row per time)
    Output: one row per customer
    """
    df=df.copy()
    # Ensure no categorical dtypes that could cause memory issues
    df["cust_id"] = df["cust_id"].astype(str)
    df["sale_id"] = df["sale_id"].astype(str)
    # Separate returned and non-returned items
    df_net = df[df["is_returned"]==0]
    
    #---- Recency: days since last purchase ----
    recency = (
        df.groupby("cust_id")["order_date"]
        .max()
        .apply(lambda x: (reference_date -x).days)
        .rename("recency_days")
    )
    #---- Frequency: number of unique orders with at least one non-returned item ---
    #sale_id is the order identifier; one order can have multiple items
    frequency = (
        df_net.groupby("cust_id")["sale_id"]
        .nunique()
        .rename("frequency")
    )
    
    # --- Is the customer a one-time buyer? ---
    # Useful because one-time buyers are less likely to return
    is_one_time_buyer = (frequency == 1).astype(int).rename("is_one_time_buyer")
    
    # --- Average and std days between orders ---
    # Only meaningful for customers with more than one order
    # Customers with a single order will get NaN -- we'll fill later with median
    order_dates_per_cust = (
        df.drop_duplicates(subset=["cust_id", "sale_id"])  # one row per order
        .sort_values(["cust_id", "order_date"])
        .groupby("cust_id")["order_date"]
    )
    
    avg_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.mean() if len(x) > 1 else np.nan)
        .rename("avg_days_between_orders")
    )
    
    std_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.std() if len(x) > 1 else np.nan)
        .rename("std_days_between_orders")
    )

    #---- Monetary: total revenue (exluding returned items)---
    total_revenue = (
        df_net.groupby("cust_id")["sale_revenue"]
        .sum()
        .rename("total_revenue")
    )
    
    #---- Average ticket: revenue per effective order  ---
    avg_ticket = (total_revenue/frequency).rename("avg_ticket")

    #---- Items per order (gross): includes returned items, captures order behavior ---
    items_per_order_gross = (
        df.groupby(["cust_id","sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("irems_per_order_gross")
    )

    #--- Items per order (net): excludes returned items, captures what they actually kept---
    items_per_order_net = (
        df_net.groupby(["cust_id","sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_net")
    )

    
    #--- Return rate: absolute count and proportion proportion of items returned---
    total_returns = (
        df.groupby("cust_id")["is_returned"]
        .sum()
        .rename("total_returns")
    )

    total_items= (
        df.groupby("cust_id")["is_returned"]
        .count()
        .rename("total_items")
    )
    return_rate = (total_returns/total_items).rename("return_rate")

    #--- Average delivery time: days between order and packing ---
    df["delivery_days"]=(df["pack_date"]-df["order_date"]).dt.days
    avg_delivery = (
        df.groupby("cust_id")["delivery_days"]
        .mean()
        .rename("avg_delivery_days")
    )

    #--- Discount behaivor ---
    df["has_discount_item"] = (df["sale_discount_applied"]<0).astype(int)

    total_discounted_items = (
        df.groupby("cust_id")["has_discount_item"]
        .sum()
        .rename("total_discounted_items")
    )

    discount_rate= (
        df.groupby("cust_id")["has_discount_item"]
        .mean()
        .rename("discount_rate")
    )
    # --- Trend features ---
    df["year"] = df["order_date"].dt.year
    
    # Proportion of orders in 2017
    orders_2017 = df[df["year"] == 2017].groupby("cust_id").size()
    orders_total = df.groupby("cust_id").size()
    pct_orders_2017 = (orders_2017 / orders_total).fillna(0).rename("pct_orders_2017")
    
    # Revenue trend: avg revenue 2017 vs 2016
    revenue_2016 = (
        df[df["year"] == 2016].groupby("cust_id")["sale_revenue"]
        .mean()
        .rename("avg_revenue_2016")
    )
    revenue_2017 = (
        df[df["year"] == 2017].groupby("cust_id")["sale_revenue"]
        .mean()
        .rename("avg_revenue_2017")
    )
    revenue_trend = (revenue_2017 - revenue_2016).rename("revenue_trend")

    
    # Combine all features into one dataframe
    rfm = pd.concat([
        recency, frequency, total_revenue, avg_ticket, items_per_order_gross, 
        items_per_order_net, total_returns, total_items, return_rate, 
        total_discounted_items, discount_rate, avg_days_between_orders, 
        std_days_between_orders, is_one_time_buyer,pct_orders_2017, revenue_trend
    ], axis=1).reset_index()
    return rfm

#Test it
rfm_train =compute_rfm_features(df_train)
print(rfm_train.shape)
rfm_train.head()
    
    

(93272, 17)


,cust_id,recency_days,frequency,total_revenue,avg_ticket,irems_per_order_gross,items_per_order_net,total_returns,total_items,return_rate,total_discounted_items,discount_rate,avg_days_between_orders,std_days_between_orders,is_one_time_buyer,pct_orders_2017,revenue_trend
0,222agnowc53dykbq,383,1.0,89.95,89.950000,1.000,1.000000,0,1,0.000000,0,0.000000,NaN,NaN,1.0,0.000000,NaN
1,222ny4m63rmalpdl,374,1.0,125.93,125.930000,3.000,2.000000,1,3,0.333333,3,1.000000,NaN,NaN,1.0,0.000000,NaN
2,223xvc4rbjatlnev,520,1.0,116.14,116.140000,2.000,2.000000,0,2,0.000000,2,1.000000,NaN,NaN,1.0,0.000000,NaN
3,223y2r357elerbis,348,1.0,47.97,47.970000,2.000,1.000000,1,2,0.500000,2,1.000000,NaN,NaN,1.0,1.000000,NaN
4,2245y4r7mgo45qg3,169,7.0,536.70,76.671429,1.125,1.142857,1,9,0.111111,8,0.888889,72.428571,51.529465,0.0,0.333333,38.143333


## Seasonality features

In [53]:
def compute_seasonality_features(df):
    """
    Computes seasonality features, aggregated at customer level.
    Captures when during the year the customer tends to buy
    Input: transaction-level dataframe
    Output: one row per customer
    """
    df = df.copy()

    #Extract month from order date
    df["order_month"]= df["order_date"].dt.month

    #--- Does the customer buy in January or July? (peak sales months)---
    buys_in_jan = (
    df.groupby("cust_id")["order_month"]
    .apply(lambda x: int((x==1).any()))
    .rename("buys_in_jan")
    )

    buys_in_jul = (
    df.groupby("cust_id")["order_month"]
    .apply(lambda x:int((x==7).any()))
    .rename("buys_in_jul")
    )

    #--- Most active month: the month where the customer placed most orders---
    #not sure about this cuz probably there will be a lot of ties
    most_active_month = (
    df.groupby(["cust_id", "order_month"])["sale_id"]
    .nunique()
    .groupby("cust_id").idxmax()
    .apply(lambda x:x[1])
    .rename("most_active_month")
    )

    #Combine
    seasonality = pd.concat([buys_in_jan, buys_in_jul, most_active_month], axis=1).reset_index()
    return seasonality

#Test it
seasonality_train = compute_seasonality_features(df_train)
print(seasonality_train.shape)
seasonality_train.head()

(93272, 4)


,cust_id,buys_in_jan,buys_in_jul,most_active_month
0,222agnowc53dykbq,0,0,12
1,222ny4m63rmalpdl,0,0,12
2,223xvc4rbjatlnev,0,1,7
3,223y2r357elerbis,1,0,1
4,2245y4r7mgo45qg3,1,1,2


## Product profile

In [54]:
def compute_product_features(df):
    """
    Computes product profile features, aggregated at customer level.
    Captures diversity of purchasing behavior across product categories,
    brands, and customer segments.
    Input: transaction-level dataframe (one row per item)
    Output: one row per customer
    """
    df = df.copy()

    # --- Number of distinct genders purchased for ---
    # A high number suggests a parent buying for the whole family
    n_genders = (
        df.groupby("cust_id")["prod_type_1"]
        .nunique()
        .rename("n_genders")
    )

    # --- Binary flags per gender ---
    buys_women = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "women").any()))
        .rename("buys_women")
    )

    buys_men = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "men").any()))
        .rename("buys_men")
    )

    buys_kids = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int(x.isin(["boys", "girls"]).any()))
        .rename("buys_kids")
    )

    # --- Diversity of product categories (prod_type_3: sneakers, boots, etc.) ---
    n_product_categories = (
        df.groupby("cust_id")["prod_type_3"]
        .nunique()
        .rename("n_product_categories")
    )

    # --- Number of distinct brands purchased ---
    n_brands = (
        df.groupby("cust_id")["prod_brand"]
        .nunique()
        .rename("n_brands")
    )

    # --- Proportion of web-only products purchased ---
    pct_web_only = (
        df.groupby("cust_id")["prod_web_only"]
        .mean()
        .rename("pct_web_only")
    )

    # --- Has the customer ever bought outlet products? ---
    buys_outlet = (
        df.groupby("cust_id")["prod_outlet"]
        .apply(lambda x: int((x > 0).any()))
        .rename("buys_outlet")
    )

    product_features = pd.concat([
        n_genders, buys_women, buys_men, buys_kids,
        n_product_categories, n_brands,
        pct_web_only, buys_outlet
    ], axis=1).reset_index()

    return product_features

# Test it
product_train = compute_product_features(df_train)
print(product_train.shape)
product_train.head()

(93272, 9)


,cust_id,n_genders,buys_women,buys_men,buys_kids,n_product_categories,n_brands,pct_web_only,buys_outlet
0,222agnowc53dykbq,1,0,1,0,1,1,0.000000,0
1,222ny4m63rmalpdl,2,0,1,1,2,2,0.000000,0
2,223xvc4rbjatlnev,1,1,0,0,1,1,0.000000,0
3,223y2r357elerbis,1,1,0,0,1,1,0.000000,0
4,2245y4r7mgo45qg3,4,1,1,1,5,7,0.222222,0


## Union

In [55]:
def build_features(df):
    """
    Builds the final customer-level feature table by combining
    all feature blocks: RFM, seasonality, and product profile.
    Input: transaction-level dataframe (one row per item)
    Output: one row per customer with all features
    """
    rfm = compute_rfm_features(df)
    seasonality = compute_seasonality_features(df)
    product = compute_product_features(df)

    # Merge all blocks on cust_id
    features = rfm.merge(seasonality, on="cust_id").merge(product, on="cust_id")

    return features

# Build features for train and valid
df_train_features = build_features(df_train)
df_valid_features = build_features(df_valid)

print("Train features shape:", df_train_features.shape)
print("Valid features shape:", df_valid_features.shape)
df_train_features.head()

Train features shape: (93272, 28)
Valid features shape: (23319, 28)


,cust_id,recency_days,frequency,total_revenue,avg_ticket,irems_per_order_gross,items_per_order_net,total_returns,total_items,return_rate,...,buys_in_jul,most_active_month,n_genders,buys_women,buys_men,buys_kids,n_product_categories,n_brands,pct_web_only,buys_outlet
0,222agnowc53dykbq,383,1.0,89.95,89.950000,1.000,1.000000,0,1,0.000000,...,0,12,1,0,1,0,1,1,0.000000,0
1,222ny4m63rmalpdl,374,1.0,125.93,125.930000,3.000,2.000000,1,3,0.333333,...,0,12,2,0,1,1,2,2,0.000000,0
2,223xvc4rbjatlnev,520,1.0,116.14,116.140000,2.000,2.000000,0,2,0.000000,...,1,7,1,1,0,0,1,1,0.000000,0
3,223y2r357elerbis,348,1.0,47.97,47.970000,2.000,1.000000,1,2,0.500000,...,0,1,1,1,0,0,1,1,0.000000,0
4,2245y4r7mgo45qg3,169,7.0,536.70,76.671429,1.125,1.142857,1,9,0.111111,...,1,2,4,1,1,1,5,7,0.222222,0


## Merge with target

In [56]:
# Merge features with targets
df_train_final = df_train_features.merge(df_train_targets, on="cust_id")
df_valid_final = df_valid_features.merge(df_valid_targets, on="cust_id")

print("Train final shape:", df_train_final.shape)
print("Valid final shape:", df_valid_final.shape)

Train final shape: (93272, 29)
Valid final shape: (23319, 29)


## Quick check

In [57]:
#Check target distribution
target = df_train_final["revenue_2018_2019"]
print("==== Target distribution ====")
print(f"Customers with revenue =0: {(target==0).sum()} ({100*(target==0).mean():.1f}%)")
print(f"Customers with revenue >0: {(target>0).sum()} ({100*(target>0).mean():.1f}%)")
print(f"\nMean (all): {target.mean():.2f}")
print(f"Mean (>0 only): {target[target>0].mean():.2f}")
print(f"Max: {target.max():.2f}")

#Check NaNs
print("\n=== NaNs per column ===")
nans= df_train_final.isna().sum()
print(nans[nans > 0])

==== Target distribution ====
Customers with revenue =0: 59196 (63.5%)
Customers with revenue >0: 34076 (36.5%)

Mean (all): 70.18
Mean (>0 only): 192.08
Max: 1197.94

=== NaNs per column ===
frequency                      7
total_revenue                  7
avg_ticket                     7
items_per_order_net            7
avg_days_between_orders    61971
std_days_between_orders    78984
is_one_time_buyer              7
revenue_trend              77146
dtype: int64


Just 7 customers returned everything they bought

In [58]:
# 1. Fill customers who returned everything
cols_to_fill = ["frequency", "total_revenue", "avg_ticket", 
                "items_per_order_net","is_one_time_buyer"]
df_train_final[cols_to_fill] = df_train_final[cols_to_fill].fillna(0)
df_valid_final[cols_to_fill] = df_valid_final[cols_to_fill].fillna(0)
df_train_final["revenue_trend"] = df_train_final["revenue_trend"].fillna(0)
df_valid_final["revenue_trend"] = df_valid_final["revenue_trend"].fillna(0)

# Fill one-time buyers (no days between orders)
for col in ["avg_days_between_orders", "std_days_between_orders"]:
    median_val = df_train_final[col].median()
    df_train_final[col] = df_train_final[col].fillna(median_val)
    df_valid_final[col] = df_valid_final[col].fillna(median_val)

# Confirm no NaNs remain
print("Remaining NaNs:", df_train_final.isna().sum().sum())

Remaining NaNs: 0


## Model preparation

In [59]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

#Define feature columns 
feature_cols = [col for col in df_train_final.columns 
                if col not in ["cust_id", "revenue_2018_2019"]]

X_train = df_train_final[feature_cols]
y_train = df_train_final["revenue_2018_2019"]

X_valid = df_valid_final[feature_cols]
y_valid = df_valid_final["revenue_2018_2019"]

print("Features:", feature_cols)
print("X_train shape:", X_train.shape)

Features: ['recency_days', 'frequency', 'total_revenue', 'avg_ticket', 'irems_per_order_gross', 'items_per_order_net', 'total_returns', 'total_items', 'return_rate', 'total_discounted_items', 'discount_rate', 'avg_days_between_orders', 'std_days_between_orders', 'is_one_time_buyer', 'pct_orders_2017', 'revenue_trend', 'buys_in_jan', 'buys_in_jul', 'most_active_month', 'n_genders', 'buys_women', 'buys_men', 'buys_kids', 'n_product_categories', 'n_brands', 'pct_web_only', 'buys_outlet']
X_train shape: (93272, 27)


## Binary target

Predict if the client will come back or not

In [60]:
# Stage 1 target: will the customer return or not? (1 = yes, 0 = no)
y_train_binary = (y_train > 0).astype(int)
y_valid_binary = (y_valid > 0).astype(int)

print("Train - will return:", y_train_binary.sum(), f"({100*y_train_binary.mean():.1f}%)")
print("Train - won't return:", (y_train_binary == 0).sum(), f"({100*(1-y_train_binary.mean()):.1f}%)")

Train - will return: 34076 (36.5%)
Train - won't return: 59196 (63.5%)


In [61]:
# Stage 1: classify whether the customer will return or not
clf = RandomForestClassifier(
    n_estimators=200, #number of trees
    max_depth=8, #max depth per tree 
    min_samples_leaf=20, #minimum samples required at each lead
    random_state=123, #seed
    n_jobs=-1 #use all available CPU cores
)
clf.fit(X_train, y_train_binary)

#Predict on validation set
y_pred_binary = clf.predict(X_valid)

#Evaluate
from sklearn.metrics import classification_report
print(classification_report(y_valid_binary, y_pred_binary))

              precision    recall  f1-score   support

           0       0.73      0.87      0.79     14737
           1       0.67      0.43      0.53      8582

    accuracy                           0.71     23319
   macro avg       0.70      0.65      0.66     23319
weighted avg       0.70      0.71      0.69     23319



In [62]:
# Add class_weight="balanced"
clf = RandomForestClassifier(
    n_estimators=200, #number of trees
    max_depth=8, #max depth per tree 
    min_samples_leaf=20, #minimum samples required at each lead
    class_weight="balanced", #penalizes mistakes on the minority class more
    random_state=123, #seed
    n_jobs=-1 #use all available CPU cores
)
clf.fit(X_train, y_train_binary)

#Predict on validation set
y_pred_binary = clf.predict(X_valid)

#Evaluate
from sklearn.metrics import classification_report
print(classification_report(y_valid_binary, y_pred_binary))

              precision    recall  f1-score   support

           0       0.75      0.78      0.76     14737
           1       0.59      0.55      0.57      8582

    accuracy                           0.69     23319
   macro avg       0.67      0.66      0.67     23319
weighted avg       0.69      0.69      0.69     23319



Better results on recall for 1

In [63]:
#Test predicting probabilities instead of hard classes
y_pred_proba =clf.predict_proba(X_valid)[:,1] #probability of returning

#Try a lower treshold
threshold= 0.4
y_pred_binary_adjusted = (y_pred_proba >= threshold).astype(int)
print(classification_report(y_valid_binary, y_pred_binary_adjusted))

              precision    recall  f1-score   support

           0       0.77      0.62      0.68     14737
           1       0.51      0.68      0.58      8582

    accuracy                           0.64     23319
   macro avg       0.64      0.65      0.63     23319
weighted avg       0.67      0.64      0.65     23319



Worst results, we will try now grid search with the balanced class weight

In [64]:
from sklearn.model_selection import GridSearchCV
#Order this better after
"""
#Define the parameter grid to search
param_grid = {
    "n_estimators": [100,200,300],
    "max_depth": [6,8,10],
    "min_samples_leaf": [10,20,30],
}

#Base classifier
clf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

#Grid search with cross validation (cv=5 means 5-fold cross validtation)
# scoring="f1" because we care about both precision and recall for class 1
grid_search = GridSearchCV(
    estimator=clf_base,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    verbose=1, #print progress
    n_jobs=-1
)

grid_search.fit(X_train, y_train_binary)
print("Best parameters:", grid_search.best_params_)
print("Best CV f1-score:", grid_search.best_score_.round(3))
"""
# Best params from grid search: max_depth=6, min_samples_leaf=20, n_estimators=200


'\n#Define the parameter grid to search\nparam_grid = {\n    "n_estimators": [100,200,300],\n    "max_depth": [6,8,10],\n    "min_samples_leaf": [10,20,30],\n}\n\n#Base classifier\nclf_base = RandomForestClassifier(\n    class_weight="balanced",\n    random_state=42,\n    n_jobs=-1\n)\n\n#Grid search with cross validation (cv=5 means 5-fold cross validtation)\n# scoring="f1" because we care about both precision and recall for class 1\ngrid_search = GridSearchCV(\n    estimator=clf_base,\n    param_grid=param_grid,\n    cv=5,\n    scoring="f1",\n    verbose=1, #print progress\n    n_jobs=-1\n)\n\ngrid_search.fit(X_train, y_train_binary)\nprint("Best parameters:", grid_search.best_params_)\nprint("Best CV f1-score:", grid_search.best_score_.round(3))\n'

In [65]:
#Train final classifier with best parameters
#clf_best = grid_search.best_estimator_
clf_best = RandomForestClassifier(
    max_depth=6,
    min_samples_leaf=20,
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
clf_best.fit(X_train, y_train_binary)

#Predict on validation set
y_pred_binary_best = clf_best.predict(X_valid)
print(classification_report(y_valid_binary, y_pred_binary_best))

              precision    recall  f1-score   support

           0       0.75      0.77      0.76     14737
           1       0.59      0.56      0.57      8582

    accuracy                           0.69     23319
   macro avg       0.67      0.67      0.67     23319
weighted avg       0.69      0.69      0.69     23319



In [66]:
# Stage 2: only train on customers who actually returned
train_returners = df_train_final[df_train_final["revenue_2018_2019"] > 0]
valid_returners = df_valid_final[df_valid_final["revenue_2018_2019"] > 0]

X_train_reg = train_returners[feature_cols]
y_train_reg = train_returners["revenue_2018_2019"]

X_valid_reg = valid_returners[feature_cols]
y_valid_reg = valid_returners["revenue_2018_2019"]

print("Train returners:", X_train_reg.shape)
print("Valid returners:", X_valid_reg.shape)

Train returners: (34076, 27)
Valid returners: (8582, 27)


In [67]:
# Stage 2: predict revenue for customers who will return
reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

reg.fit(X_train_reg, y_train_reg)

# Evaluate on returners only first
y_pred_reg = reg.predict(X_valid_reg)

mae_returners = mean_absolute_error(y_valid_reg, y_pred_reg)
spearman_returners = spearmanr(y_valid_reg, y_pred_reg).statistic

print(f"MAE (returners only):      {mae_returners:.2f}")
print(f"Spearman (returners only): {spearman_returners:.3f}")

MAE (returners only):      117.12
Spearman (returners only): 0.348


In [68]:
# Step 1: predict who will return using the classifier
y_pred_proba = clf_best.predict_proba(X_valid)[:, 1]
y_pred_will_return = (y_pred_proba >= 0.5).astype(int)

# Step 2: for predicted returners, predict revenue using the regressor
y_final_pred = np.zeros(len(X_valid))  # start everyone at 0

returner_mask = y_pred_will_return == 1
y_final_pred[returner_mask] = reg.predict(X_valid[returner_mask])

# Evaluate full pipeline
mae_full = mean_absolute_error(y_valid, y_final_pred)
spearman_full = spearmanr(y_valid, y_final_pred).statistic

print(f"MAE (full pipeline):      {mae_full:.2f}")
print(f"Spearman (full pipeline): {spearman_full:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (full pipeline):      80.24
Spearman (full pipeline): 0.400

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [69]:
from sklearn.model_selection import GridSearchCV

#Order after
"""
param_grid_reg = {
    "n_estimators": [100, 200, 300],
    "max_depth": [6, 8, 10],
    "min_samples_leaf": [10, 20, 30],
}

reg_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# scoring="neg_mean_absolute_error" because GridSearchCV maximizes the score
# so we use the negative MAE (less negative = better)
grid_search_reg = GridSearchCV(
    estimator=reg_base,
    param_grid=param_grid_reg,
    cv=5,
    scoring="neg_mean_absolute_error",
    verbose=1,
    n_jobs=-1
)

grid_search_reg.fit(X_train_reg, y_train_reg)

print("Best parameters:", grid_search_reg.best_params_)
print("Best CV MAE:", -grid_search_reg.best_score_.round(2))
"""
# Best params from grid search: max_depth=6, min_samples_leaf=30, n_estimators=300

'\nparam_grid_reg = {\n    "n_estimators": [100, 200, 300],\n    "max_depth": [6, 8, 10],\n    "min_samples_leaf": [10, 20, 30],\n}\n\nreg_base = RandomForestRegressor(\n    random_state=42,\n    n_jobs=-1\n)\n\n# scoring="neg_mean_absolute_error" because GridSearchCV maximizes the score\n# so we use the negative MAE (less negative = better)\ngrid_search_reg = GridSearchCV(\n    estimator=reg_base,\n    param_grid=param_grid_reg,\n    cv=5,\n    scoring="neg_mean_absolute_error",\n    verbose=1,\n    n_jobs=-1\n)\n\ngrid_search_reg.fit(X_train_reg, y_train_reg)\n\nprint("Best parameters:", grid_search_reg.best_params_)\nprint("Best CV MAE:", -grid_search_reg.best_score_.round(2))\n'

In [70]:
# Get best regressor
#reg_best = grid_search_reg.best_estimator_

# Best params from grid search: max_depth=6, min_samples_leaf=10, n_estimators=200
reg_best = RandomForestRegressor(
    max_depth=6,
    min_samples_leaf=10,
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
reg_best.fit(X_train_reg, y_train_reg)

# Full pipeline with optimized regressor
y_final_pred_v2 = np.zeros(len(X_valid))
y_final_pred_v2[returner_mask] = reg_best.predict(X_valid[returner_mask])
mae_v2 = mean_absolute_error(y_valid, y_final_pred_v2)
spearman_v2 = spearmanr(y_valid, y_final_pred_v2).statistic
print(f"MAE (full pipeline v2):      {mae_v2:.2f}")
print(f"Spearman (full pipeline v2): {spearman_v2:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")


MAE (full pipeline v2):      80.28
Spearman (full pipeline v2): 0.399

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [71]:
from xgboost import XGBClassifier, XGBRegressor

# --- Stage 1: XGBoost Classifier ---
# scale_pos_weight handles class imbalance (equivalent to class_weight="balanced")
# it's calculated as: number of negatives / number of positives
scale = (y_train_binary == 0).sum() / (y_train_binary == 1).sum()
print(f"scale_pos_weight: {scale:.2f}")

clf_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,     # how much each tree contributes — lower = more conservative
    subsample=0.8,          # fraction of samples used per tree — reduces overfitting
    colsample_bytree=0.8,   # fraction of features used per tree — reduces overfitting
    scale_pos_weight=scale, # handles class imbalance
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

clf_xgb.fit(X_train, y_train_binary)
y_pred_binary_xgb = clf_xgb.predict(X_valid)
print(classification_report(y_valid_binary, y_pred_binary_xgb))

scale_pos_weight: 1.74
              precision    recall  f1-score   support

           0       0.75      0.77      0.76     14737
           1       0.59      0.56      0.57      8582

    accuracy                           0.69     23319
   macro avg       0.67      0.66      0.67     23319
weighted avg       0.69      0.69      0.69     23319



In [72]:
# --- Stage 2: XGBoost Regressor ---
reg_xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

reg_xgb.fit(X_train_reg, y_train_reg)

# --- Full pipeline with XGBoost ---
y_pred_proba_xgb = clf_xgb.predict_proba(X_valid)[:, 1]
returner_mask_xgb = (y_pred_proba_xgb >= 0.5)

y_final_pred_xgb = np.zeros(len(X_valid))
y_final_pred_xgb[returner_mask_xgb] = reg_xgb.predict(X_valid[returner_mask_xgb])

mae_xgb = mean_absolute_error(y_valid, y_final_pred_xgb)
spearman_xgb = spearmanr(y_valid, y_final_pred_xgb).statistic

print(f"MAE (XGBoost pipeline):      {mae_xgb:.2f}")
print(f"Spearman (XGBoost pipeline): {spearman_xgb:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (XGBoost pipeline):      79.99
Spearman (XGBoost pipeline): 0.398

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [73]:
# Log transform the target (add 1 to avoid log(0))
# We use log1p which is log(1+x) -- a common trick for revenue/count data
y_train_reg_log = np.log1p(y_train_reg)
y_valid_reg_log = np.log1p(y_valid_reg)

# Train XGBoost regressor on log-transformed target
reg_xgb_log = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

reg_xgb_log.fit(X_train_reg, y_train_reg_log)

# Predict and convert back with expm1 (inverse of log1p)
y_pred_reg_log = np.expm1(reg_xgb_log.predict(X_valid_reg))

mae_log = mean_absolute_error(y_valid_reg, y_pred_reg_log)
spearman_log = spearmanr(y_valid_reg, y_pred_reg_log).statistic

print(f"MAE (log transform, returners only): {mae_log:.2f}")
print(f"Spearman (log transform, returners only): {spearman_log:.3f}")

MAE (log transform, returners only): 110.68
Spearman (log transform, returners only): 0.349


In [74]:
y_final_pred_log = np.zeros(len(X_valid))
y_final_pred_log[returner_mask_xgb] = np.expm1(
    reg_xgb_log.predict(X_valid[returner_mask_xgb])
)

mae_log_full = mean_absolute_error(y_valid, y_final_pred_log)
spearman_log_full = spearmanr(y_valid, y_final_pred_log).statistic

print(f"MAE (log pipeline):      {mae_log_full:.2f}")
print(f"Spearman (log pipeline): {spearman_log_full:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (log pipeline):      71.17
Spearman (log pipeline): 0.397

Target to beat -> MAE: 61.15 | Spearman: 0.41


## Trying feature tools

In [75]:
import featuretools as ft
import warnings
warnings.filterwarnings("ignore")
# Create an EntitySet -- this is how featuretools understands your data
es = ft.EntitySet(id="clv_data")

# Add the transactions table
es = es.add_dataframe(
    dataframe_name="transactions",
    dataframe=df_train,  
    index="index",       # featuretools needs a unique index
    make_index=True,     # create one automatically
    time_index="order_date",
    logical_types={
        "cust_id": "Categorical",
        "sale_id": "Categorical",
        "prod_id": "Categorical",
        "prod_brand": "Categorical",
        "prod_type_1": "Categorical",
        "prod_type_3": "Categorical",
    }
)

# Add the customers table
customers = df_train[["cust_id"]].drop_duplicates().reset_index(drop=True)
es = es.add_dataframe(
    dataframe_name="customers",
    dataframe=customers,
    index="cust_id",
)

# Define the relationship: one customer has many transactions
es = es.add_relationship("customers", "cust_id", "transactions", "cust_id")

print(es)

Entityset: clv_data
  DataFrames:
    transactions [Rows: 219128, Columns: 28]
    customers [Rows: 93272, Columns: 1]
  Relationships:
    transactions.cust_id -> customers.cust_id


In [76]:
# Run Deep Feature Synthesis
# agg_primitives: operations applied across all transactions of a customer
# trans_primitives: operations applied to individual columns
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="customers",  # we want one row per customer
    agg_primitives=[
        "sum", "mean", "max", "min", "std", "count",
        "percent_true", "num_unique", "last"
    ],
    trans_primitives=[],  # keep it simple for now
    max_depth=1,          # depth 1 = one level of aggregation, faster
    verbose=True
)

print("Feature matrix shape:", feature_matrix.shape)
print("Number of features generated:", len(feature_defs))
feature_matrix.head()

Built 78 features
Elapsed: 01:11 | Progress: 100%|████████████████████████████████████████████████
Feature matrix shape: (93272, 78)
Number of features generated: 78


,COUNT(transactions),LAST(transactions.index),LAST(transactions.is_returned),LAST(transactions.prod_brand),LAST(transactions.prod_clasp),LAST(transactions.prod_color),LAST(transactions.prod_comfort_sole),LAST(transactions.prod_comfort_wear),LAST(transactions.prod_heel),LAST(transactions.prod_id),...,STD(transactions.prod_web_only),STD(transactions.sale_discount_applied),STD(transactions.sale_revenue),SUM(transactions.is_returned),SUM(transactions.prod_insole),SUM(transactions.prod_outlet),SUM(transactions.prod_size),SUM(transactions.prod_web_only),SUM(transactions.sale_discount_applied),SUM(transactions.sale_revenue)
cust_id,,,,,,,,,,,,,,,,,,,,,
klantwj2374mzmab,2,1,1,Mario Rossi,zipper,cognac,NaN,NaN,NaN,7ewnqhtrquent4cy,...,0.0,3.026417,58.901995,1.0,1.0,0.0,74.0,0.0,-75.68,83.30
a63atwr2ig2jfprr,4,153282,0,Geox,velcro,white,NaN,breathable,<2.5 cm,ncu77k3laiu4ocs3,...,0.5,7.764347,26.980000,3.0,0.0,0.0,102.5,1.0,-64.95,53.96
zr7ihbfbi6gcy2tz,2,103134,0,Geox,"elastic, velcro",dark blue,"anti-shock, pressure relieving",breathable,NaN,354alzzzqljbzvpe,...,0.0,9.715647,2.644579,0.0,1.0,0.0,72.0,0.0,-66.20,83.70
dt7cthjqnjmkbiu6,4,142789,0,K-Swiss,laces,white,NaN,NaN,NaN,untziwcuqmcgybl6,...,0.5,17.308961,80.668027,1.0,1.0,0.0,138.0,1.0,-59.96,350.92
etcrrgcbrzfovyzj,6,105476,0,Gosh,zipper,brown,NaN,NaN,5-8 cm,lbgxuiskcury3jeh,...,0.0,26.869592,33.586332,3.0,0.0,0.0,221.0,0.0,-715.50,144.50


In [77]:
# Reset index to get cust_id as a column
feature_matrix = feature_matrix.reset_index()

# Merge with targets
df_train_ft = feature_matrix.merge(df_train_targets, on="cust_id")

# Check NaNs
nans_ft = df_train_ft.isna().sum()
print("Columns with NaNs:")
print(nans_ft[nans_ft > 0])
print("\nShape:", df_train_ft.shape)

Columns with NaNs:
LAST(transactions.prod_clasp)               9506
LAST(transactions.prod_comfort_sole)       80283
LAST(transactions.prod_comfort_wear)       84176
LAST(transactions.prod_heel)               35529
LAST(transactions.prod_insole)              5778
LAST(transactions.prod_material)            5037
LAST(transactions.prod_print)              73146
LAST(transactions.prod_type_4)             52241
LAST(transactions.prod_type_5)              1428
LAST(transactions.returned_to_shop_id)     86160
MAX(transactions.prod_insole)               4477
MEAN(transactions.prod_insole)              4477
MIN(transactions.prod_insole)               4477
STD(transactions.is_returned)              49347
STD(transactions.prod_insole)              52310
STD(transactions.prod_outlet)              49347
STD(transactions.prod_size)                49347
STD(transactions.prod_web_only)            49347
STD(transactions.sale_discount_applied)    49347
STD(transactions.sale_revenue)             49347
d

In [78]:
# Convert LAST categorical columns to string first, then fill
last_cols = [col for col in df_train_ft.columns if col.startswith("LAST(")]
for col in last_cols:
    df_train_ft[col] = df_train_ft[col].astype(str).fillna("unknown")

# Fill STD columns with 0 (one-time buyers have no std)
std_cols = [col for col in df_train_ft.columns if col.startswith("STD(")]
df_train_ft[std_cols] = df_train_ft[std_cols].fillna(0)

# Fill remaining numeric NaNs with 0
num_cols = df_train_ft.select_dtypes(include=[np.number]).columns
df_train_ft[num_cols] = df_train_ft[num_cols].fillna(0)

# Confirm
print("Remaining NaNs:", df_train_ft.isna().sum().sum())

Remaining NaNs: 0


In [79]:
# Apply EXACTLY the same features learned from train to valid
feature_matrix_valid = ft.calculate_feature_matrix(
    features=feature_defs,  # same feature definitions from train
    entityset=es_valid,
    verbose=True
)

feature_matrix_valid = feature_matrix_valid.reset_index()

# Same NaN cleaning
for col in [c for c in feature_matrix_valid.columns if c.startswith("LAST(")]:
    feature_matrix_valid[col] = feature_matrix_valid[col].astype(str).fillna("unknown")

std_cols_valid = [c for c in feature_matrix_valid.columns if c.startswith("STD(")]
feature_matrix_valid[std_cols_valid] = feature_matrix_valid[std_cols_valid].fillna(0)

num_cols_valid = feature_matrix_valid.select_dtypes(include=[np.number]).columns
feature_matrix_valid[num_cols_valid] = feature_matrix_valid[num_cols_valid].fillna(0)

# Merge with targets
df_valid_ft = feature_matrix_valid.merge(df_valid_targets, on="cust_id")

print("Valid shape:", df_valid_ft.shape)
print("Remaining NaNs:", df_valid_ft.isna().sum().sum())

Elapsed: 00:18 | Progress: 100%|████████████████████████████████████████████████
Valid shape: (23319, 80)
Remaining NaNs: 0


In [80]:
# Drop LAST columns -- they are strings and less informative than aggregations
last_cols = [col for col in df_train_ft.columns if col.startswith("LAST(")]
df_train_ft = df_train_ft.drop(columns=last_cols)
df_valid_ft = df_valid_ft.drop(columns=last_cols)

# Redefine feature columns
feature_cols_ft = [col for col in df_train_ft.columns 
                   if col not in ["cust_id", "revenue_2018_2019"]]

X_train_ft = df_train_ft[feature_cols_ft]
y_train_ft = df_train_ft["revenue_2018_2019"]

X_valid_ft = df_valid_ft[feature_cols_ft]
y_valid_ft = df_valid_ft["revenue_2018_2019"]

# Binary target for classifier
y_train_ft_binary = (y_train_ft > 0).astype(int)
y_valid_ft_binary = (y_valid_ft > 0).astype(int)

In [81]:
# Stage 1: Classifier
clf_ft = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train_ft_binary == 0).sum() / (y_train_ft_binary == 1).sum(),
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
clf_ft.fit(X_train_ft, y_train_ft_binary)

# Stage 2: Regressor (only on returners)
train_returners_ft = df_train_ft[df_train_ft["revenue_2018_2019"] > 0]
X_train_ft_reg = train_returners_ft[feature_cols_ft]
y_train_ft_reg = np.log1p(train_returners_ft["revenue_2018_2019"])

reg_ft = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
reg_ft.fit(X_train_ft_reg, y_train_ft_reg)

# Full pipeline
y_pred_proba_ft = clf_ft.predict_proba(X_valid_ft)[:, 1]
returner_mask_ft = (y_pred_proba_ft >= 0.5)

y_final_pred_ft = np.zeros(len(X_valid_ft))
y_final_pred_ft[returner_mask_ft] = np.expm1(reg_ft.predict(X_valid_ft[returner_mask_ft]))

mae_ft = mean_absolute_error(y_valid_ft, y_final_pred_ft)
spearman_ft = spearmanr(y_valid_ft, y_final_pred_ft).statistic

print(f"MAE (FeatureTools pipeline):      {mae_ft:.2f}")
print(f"Spearman (FeatureTools pipeline): {spearman_ft:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (FeatureTools pipeline):      72.30
Spearman (FeatureTools pipeline): 0.389

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [82]:
# Combine manual features with featuretools features
ft_only_cols = [col for col in df_train_ft.columns 
                if col not in ["cust_id", "revenue_2018_2019"]]

df_train_combined = df_train_final.merge(
    df_train_ft[["cust_id"] + ft_only_cols], on="cust_id"
)
df_valid_combined = df_valid_final.merge(
    df_valid_ft[["cust_id"] + ft_only_cols], on="cust_id"
)

feature_cols_combined = [col for col in df_train_combined.columns 
                         if col not in ["cust_id", "revenue_2018_2019"]]

print("Combined features:", len(feature_cols_combined))
print("Train shape:", df_train_combined.shape)

Combined features: 80
Train shape: (93272, 82)


In [83]:
# Combined features
X_train_comb = df_train_combined[feature_cols_combined]
y_train_comb = df_train_combined["revenue_2018_2019"]

X_valid_comb = df_valid_combined[feature_cols_combined]
y_valid_comb = df_valid_combined["revenue_2018_2019"]

y_train_comb_binary = (y_train_comb > 0).astype(int)
y_valid_comb_binary = (y_valid_comb > 0).astype(int)

# Stage 1: Classifier
clf_comb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train_comb_binary == 0).sum() / (y_train_comb_binary == 1).sum(),
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
clf_comb.fit(X_train_comb, y_train_comb_binary)

# Stage 2: Regressor
train_returners_comb = df_train_combined[df_train_combined["revenue_2018_2019"] > 0]
X_train_comb_reg = train_returners_comb[feature_cols_combined]
y_train_comb_reg = np.log1p(train_returners_comb["revenue_2018_2019"])

reg_comb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
reg_comb.fit(X_train_comb_reg, y_train_comb_reg)

# Full pipeline
y_pred_proba_comb = clf_comb.predict_proba(X_valid_comb)[:, 1]
returner_mask_comb = (y_pred_proba_comb >= 0.5)

y_final_pred_comb = np.zeros(len(X_valid_comb))
y_final_pred_comb[returner_mask_comb] = np.expm1(
    reg_comb.predict(X_valid_comb[returner_mask_comb])
)

mae_comb = mean_absolute_error(y_valid_comb, y_final_pred_comb)
spearman_comb = spearmanr(y_valid_comb, y_final_pred_comb).statistic

print(f"MAE (combined pipeline):      {mae_comb:.2f}")
print(f"Spearman (combined pipeline): {spearman_comb:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (combined pipeline):      71.13
Spearman (combined pipeline): 0.403

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [84]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

param_grid_xgb = {
    "n_estimators": [200, 300, 500],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8],
}

reg_xgb_base = XGBRegressor(
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

grid_search_xgb = GridSearchCV(
    estimator=reg_xgb_base,
    param_grid=param_grid_xgb,
    cv=5,
    scoring="neg_mean_absolute_error",
    verbose=1,
    n_jobs=-1
)

grid_search_xgb.fit(X_train_reg, y_train_reg_log)

print("Best parameters:", grid_search_xgb.best_params_)
print("Best CV MAE:", -grid_search_xgb.best_score_.round(2))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 500, 'subsample': 0.7}
Best CV MAE: 0.64


In [85]:
reg_xgb_best = grid_search_xgb.best_estimator_

# Full pipeline
y_pred_proba_xgb2 = clf_xgb.predict_proba(X_valid)[:, 1]
returner_mask_xgb2 = (y_pred_proba_xgb2 >= 0.5)

y_final_pred_xgb2 = np.zeros(len(X_valid))
y_final_pred_xgb2[returner_mask_xgb2] = np.expm1(
    reg_xgb_best.predict(X_valid[returner_mask_xgb2])
)

mae_xgb2 = mean_absolute_error(y_valid, y_final_pred_xgb2)
spearman_xgb2 = spearmanr(y_valid, y_final_pred_xgb2).statistic

print(f"MAE (XGBoost optimizado):      {mae_xgb2:.2f}")
print(f"Spearman (XGBoost optimizado): {spearman_xgb2:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (XGBoost optimizado):      70.98
Spearman (XGBoost optimizado): 0.400

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [86]:
from lightgbm import LGBMClassifier, LGBMRegressor

# --- Stage 1: LightGBM Classifier ---
scale = (y_train_binary == 0).sum() / (y_train_binary == 1).sum()

clf_lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    verbose=-1        # silence output
)
clf_lgbm.fit(X_train, y_train_binary)

# --- Stage 2: LightGBM Regressor ---
reg_lgbm = LGBMRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,  # same best params we found for XGBoost
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
reg_lgbm.fit(X_train_reg, y_train_reg_log)

# --- Full pipeline ---
y_pred_proba_lgbm = clf_lgbm.predict_proba(X_valid)[:, 1]
returner_mask_lgbm = (y_pred_proba_lgbm >= 0.5)

y_final_pred_lgbm = np.zeros(len(X_valid))
y_final_pred_lgbm[returner_mask_lgbm] = np.expm1(
    reg_lgbm.predict(X_valid[returner_mask_lgbm])
)

mae_lgbm = mean_absolute_error(y_valid, y_final_pred_lgbm)
spearman_lgbm = spearmanr(y_valid, y_final_pred_lgbm).statistic

print(f"MAE (LightGBM pipeline):      {mae_lgbm:.2f}")
print(f"Spearman (LightGBM pipeline): {spearman_lgbm:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (LightGBM pipeline):      71.03
Spearman (LightGBM pipeline): 0.399

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [87]:
# Direct regression: predict revenue for all customers at once
reg_direct = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# Train on ALL customers (including zeros)
reg_direct.fit(X_train, np.log1p(y_train))

# Predict and convert back
y_pred_direct = np.expm1(reg_direct.predict(X_valid))

# Clip negative predictions to 0 (log transform can sometimes give slightly negative values)
y_pred_direct = np.clip(y_pred_direct, 0, None)

mae_direct = mean_absolute_error(y_valid, y_pred_direct)
spearman_direct = spearmanr(y_valid, y_pred_direct).statistic

print(f"MAE (direct regression):      {mae_direct:.2f}")
print(f"Spearman (direct regression): {spearman_direct:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (direct regression):      66.22
Spearman (direct regression): 0.392

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [88]:
param_grid_direct = {
    "n_estimators": [300, 500, 700],
    "max_depth": [3, 4, 6],
    "learning_rate": [0.01, 0.05],
    "subsample": [0.7, 0.8],
}

reg_direct_base = XGBRegressor(
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

grid_search_direct = GridSearchCV(
    estimator=reg_direct_base,
    param_grid=param_grid_direct,
    cv=5,
    scoring="neg_mean_absolute_error",
    verbose=1,
    n_jobs=-1
)

# Train on ALL customers with log transform
grid_search_direct.fit(X_train, np.log1p(y_train))

print("Best parameters:", grid_search_direct.best_params_)
print("Best CV MAE:", -grid_search_direct.best_score_.round(4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 700, 'subsample': 0.7}
Best CV MAE: 1.8641


In [89]:
reg_direct_best = grid_search_direct.best_estimator_

# Predict and convert back
y_pred_direct_best = np.expm1(reg_direct_best.predict(X_valid))
y_pred_direct_best = np.clip(y_pred_direct_best, 0, None)

mae_direct_best = mean_absolute_error(y_valid, y_pred_direct_best)
spearman_direct_best = spearmanr(y_valid, y_pred_direct_best).statistic

print(f"MAE (direct regression optimizado):      {mae_direct_best:.2f}")
print(f"Spearman (direct regression optimizado): {spearman_direct_best:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")


MAE (direct regression optimizado):      65.91
Spearman (direct regression optimizado): 0.391

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [90]:
# Create a dataframe with predictions vs actual
error_analysis = df_valid_final[["cust_id", "revenue_2018_2019"]].copy()
error_analysis["predicted"] = y_pred_direct_best
error_analysis["error"] = np.abs(error_analysis["revenue_2018_2019"] - error_analysis["predicted"])
error_analysis["error_signed"] = error_analysis["revenue_2018_2019"] - error_analysis["predicted"]

error_analysis = error_analysis.sort_values("error", ascending=False)

print("=== Top 20 biggest errors ===")
print(error_analysis.head(20))

print(f"\n=== Error distribution ===")
print(f"Customers with error > 500:  {(error_analysis['error'] > 500).sum()}")
print(f"Customers with error > 200:  {(error_analysis['error'] > 200).sum()}")
print(f"Customers with error > 100:  {(error_analysis['error'] > 100).sum()}")
print(f"Customers with error < 50:   {(error_analysis['error'] < 50).sum()}")

print(f"\n=== Signed error ===")
print(f"Underpredicting (actual > pred): {(error_analysis['error_signed'] > 0).sum()}")
print(f"Overpredicting (pred > actual):  {(error_analysis['error_signed'] < 0).sum()}")

=== Top 20 biggest errors ===
                cust_id  revenue_2018_2019   predicted        error  \
20044  vkkljb27ovnlzde6            1185.98    3.038321  1182.941679   
21639  xonhxxf3ijmyvqdv            1175.05    2.493662  1172.556338   
13581  mn5oibpoumhj4zml            1138.07    5.819933  1132.250067   
4100   7pmkwtfvwhtfzb3c            1168.70   41.953690  1126.746310   
2251   56fmop2mqsio4jrf            1109.65    2.958333  1106.691667   
19736  v2x73lef5pqu46zz            1103.16    3.984269  1099.175731   
4854   ar7fp4pg57rcw3jw            1105.34    7.179793  1098.160207   
5810   byu3nvqddsbw4nuw            1097.90    7.568835  1090.331165   
16190  q5byopumbg7szitu            1092.09    6.923223  1085.166777   
19523  uq62swkbvjinp7s3            1089.67    4.740839  1084.929161   
879    37mohe3bfvdbryyd            1084.43    8.565829  1075.864171   
18252  swiih5u33tokvci3            1100.68   51.824177  1048.855823   
16789  qwupwyqpzz626omh            1054.29   19

In [91]:
# Who are the high-value customers the model is missing?
high_error = error_analysis[error_analysis["error"] > 500].copy()

# Merge with their features to understand them
high_error_features = high_error.merge(df_valid_final, on="cust_id")

print("=== High error customers (error > 500) ===")
print(f"Count: {len(high_error_features)}")
print(f"\nAverage features vs rest:")

rest = df_valid_final[~df_valid_final["cust_id"].isin(high_error["cust_id"])]

cols_to_compare = ["frequency", "total_revenue", "avg_ticket", 
                   "return_rate", "recency_days", "pct_orders_2017"]

for col in cols_to_compare:
    print(f"{col:30s} high_error: {high_error_features[col].mean():.2f} | rest: {rest[col].mean():.2f}")

=== High error customers (error > 500) ===
Count: 471

Average features vs rest:
frequency                      high_error: 3.00 | rest: 1.49
total_revenue                  high_error: 297.17 | rest: 122.41
avg_ticket                     high_error: 103.43 | rest: 80.64
return_rate                    high_error: 0.18 | rest: 0.09
recency_days                   high_error: 182.22 | rest: 300.76
pct_orders_2017                high_error: 0.69 | rest: 0.58


In [92]:
# VIP customer: high frequency and high revenue in 2016-2017
df_train_final["is_vip"] = (
    (df_train_final["frequency"] >= 3) & 
    (df_train_final["total_revenue"] >= 200)
).astype(int)

df_valid_final["is_vip"] = (
    (df_valid_final["frequency"] >= 3) & 
    (df_valid_final["total_revenue"] >= 200)
).astype(int)

print("VIP customers in train:", df_train_final["is_vip"].sum())
print("VIP customers in valid:", df_valid_final["is_vip"].sum())

VIP customers in train: 8729
VIP customers in valid: 2243


In [93]:
print("=== Frequency distribution ===")
print(df_train_final["frequency"].describe())
print(df_train_final["frequency"].value_counts().head(10))

print("\n=== Total revenue distribution (customers who returned) ===")
returners = df_train_final[df_train_final["total_revenue"] > 0]
print(returners["total_revenue"].describe())
print(f"\nTop 5%:  {returners['total_revenue'].quantile(0.95):.2f}")
print(f"Top 10%: {returners['total_revenue'].quantile(0.90):.2f}")

=== Frequency distribution ===
count    93272.000000
mean         1.522375
std          1.138788
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         23.000000
Name: frequency, dtype: float64
frequency
1.0     67214
2.0     15104
3.0      5548
4.0      2582
5.0      1275
6.0       671
7.0       394
8.0       202
9.0       126
10.0       62
Name: count, dtype: int64

=== Total revenue distribution (customers who returned) ===
count    93265.000000
mean       124.963880
std        118.217544
min          9.990000
25%         59.300000
50%         89.900000
75%        145.000000
max       1195.800000
Name: total_revenue, dtype: float64

Top 5%:  347.19
Top 10%: 249.00


In [94]:
# Create VIP feature
df_train_final["is_vip"] = (
    (df_train_final["frequency"] >= 4) & 
    (df_train_final["total_revenue"] >= 249)
).astype(int)

df_valid_final["is_vip"] = (
    (df_valid_final["frequency"] >= 4) & 
    (df_valid_final["total_revenue"] >= 249)
).astype(int)

print("VIP customers in train:", df_train_final["is_vip"].sum())
print("VIP customers in valid:", df_valid_final["is_vip"].sum())

# Redefine feature cols now that we have is_vip
feature_cols = [col for col in df_train_final.columns 
                if col not in ["cust_id", "revenue_2018_2019"]]

X_train = df_train_final[feature_cols]
X_valid = df_valid_final[feature_cols]

VIP customers in train: 4677
VIP customers in valid: 1202


In [95]:
reg_vip = XGBRegressor(
    n_estimators=700,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
reg_vip.fit(X_train, np.log1p(y_train))

y_pred_vip = np.expm1(reg_vip.predict(X_valid))
y_pred_vip = np.clip(y_pred_vip, 0, None)

mae_vip = mean_absolute_error(y_valid, y_pred_vip)
spearman_vip = spearmanr(y_valid, y_pred_vip).statistic

print(f"MAE (VIP feature):      {mae_vip:.2f}")
print(f"Spearman (VIP feature): {spearman_vip:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

MAE (VIP feature):      65.87
Spearman (VIP feature): 0.391

Target to beat -> MAE: 61.15 | Spearman: 0.41


### Best MAE until now

### Trying clustering

In [97]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Use RFM features for clustering
rfm_cols = ["recency_days", "frequency", "total_revenue"]

scaler = StandardScaler()
X_rfm = scaler.fit_transform(df_train_final[rfm_cols])

# Try different number of clusters to find optimal
inertias = []
for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_rfm)
    inertias.append((k, kmeans.inertia_))
    print(f"k={k}, inertia={kmeans.inertia_:.0f}")

print("\nDone! Pick the k where inertia stops dropping fast (elbow point)")

k=2, inertia=163377
k=3, inertia=100483
k=4, inertia=72996
k=5, inertia=61034
k=6, inertia=51018
k=7, inertia=45974

Done! Pick the k where inertia stops dropping fast (elbow point)


In [98]:
# Train final KMeans with k=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df_train_final["cluster"] = kmeans_final.fit_predict(
    scaler.transform(df_train_final[rfm_cols])
)
df_valid_final["cluster"] = kmeans_final.predict(
    scaler.transform(df_valid_final[rfm_cols])
)

# Understand each cluster
print("=== Cluster profiles ===")
print(df_train_final.groupby("cluster")[rfm_cols + ["revenue_2018_2019"]].mean().round(2))
print("\nCluster sizes:")
print(df_train_final["cluster"].value_counts().sort_index())

=== Cluster profiles ===
         recency_days  frequency  total_revenue  revenue_2018_2019
cluster                                                           
0              172.08       2.85         259.20             155.22
1              553.07       1.10          85.87              32.41
2              188.06       1.16          85.66              53.88
3              110.70       6.10         596.25             353.90

Cluster sizes:
cluster
0    13029
1    28961
2    48544
3     2738
Name: count, dtype: int64


In [99]:
# Train one model per cluster
cluster_models = {}

for cluster_id in range(4):
    # Get train data for this cluster
    train_cluster = df_train_final[df_train_final["cluster"] == cluster_id]
    X_c = train_cluster[feature_cols]
    y_c = np.log1p(train_cluster["revenue_2018_2019"])
    
    model = XGBRegressor(
        n_estimators=700,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.7,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
    model.fit(X_c, y_c)
    cluster_models[cluster_id] = model
    print(f"Cluster {cluster_id} trained — {len(train_cluster)} customers")

# Predict on valid set using the corresponding model per customer
y_pred_cluster = np.zeros(len(df_valid_final))
for cluster_id in range(4):
    mask = df_valid_final["cluster"] == cluster_id
    if mask.sum() > 0:
        y_pred_cluster[mask] = np.expm1(
            cluster_models[cluster_id].predict(X_valid[mask])
        )

y_pred_cluster = np.clip(y_pred_cluster, 0, None)

mae_cluster = mean_absolute_error(y_valid, y_pred_cluster)
spearman_cluster = spearmanr(y_valid, y_pred_cluster).statistic

print(f"\nMAE (clustering):      {mae_cluster:.2f}")
print(f"Spearman (clustering): {spearman_cluster:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

Cluster 0 trained — 13029 customers
Cluster 1 trained — 28961 customers
Cluster 2 trained — 48544 customers
Cluster 3 trained — 2738 customers

MAE (clustering):      66.01
Spearman (clustering): 0.385

Target to beat -> MAE: 61.15 | Spearman: 0.41


In [100]:
# Cluster with more features
cluster_cols = [
    "recency_days", "frequency", "total_revenue", 
    "return_rate", "avg_ticket", "pct_orders_2017",
    "n_genders", "avg_days_between_orders"
]

# Need to fill NaN in avg_days_between_orders before scaling
X_cluster = df_train_final[cluster_cols].fillna(0)
X_cluster_valid = df_valid_final[cluster_cols].fillna(0)

scaler2 = StandardScaler()
X_cluster_scaled = scaler2.fit_transform(X_cluster)
X_cluster_valid_scaled = scaler2.transform(X_cluster_valid)

kmeans2 = KMeans(n_clusters=4, random_state=42, n_init=10)
df_train_final["cluster2"] = kmeans2.fit_predict(X_cluster_scaled)
df_valid_final["cluster2"] = kmeans2.predict(X_cluster_valid_scaled)

print("=== Cluster profiles ===")
print(df_train_final.groupby("cluster2")[cluster_cols + ["revenue_2018_2019"]].mean().round(2))
print("\nCluster sizes:")
print(df_train_final["cluster2"].value_counts().sort_index())

=== Cluster profiles ===
          recency_days  frequency  total_revenue  return_rate  avg_ticket  \
cluster2                                                                    
0               186.05       1.18          91.00         0.07       78.14   
1               546.40       1.13          89.09         0.06       79.15   
2               155.05       4.07         377.79         0.19       99.65   
3               157.87       1.96         158.81         0.18       81.57   

          pct_orders_2017  n_genders  avg_days_between_orders  \
cluster2                                                        
0                    0.98       1.11                   111.49   
1                    0.00       1.09                   111.51   
2                    0.61       2.26                   110.26   
3                    0.52       1.53                   341.77   

          revenue_2018_2019  
cluster2                     
0                     54.82  
1                     34.08  
2

In [101]:
inertias2 = []
for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias2.append((k, kmeans.inertia_))
    print(f"k={k}, inertia={kmeans.inertia_:.0f}")
    

k=2, inertia=586265
k=3, inertia=449999
k=4, inertia=381800
k=5, inertia=328598
k=6, inertia=293085
k=7, inertia=266237


In [102]:
kmeans2 = KMeans(n_clusters=5, random_state=42, n_init=10)
df_train_final["cluster2"] = kmeans2.fit_predict(X_cluster_scaled)
df_valid_final["cluster2"] = kmeans2.predict(X_cluster_valid_scaled)

print("=== Cluster profiles ===")
print(df_train_final.groupby("cluster2")[cluster_cols + ["revenue_2018_2019"]].mean().round(2))
print("\nCluster sizes:")
print(df_train_final["cluster2"].value_counts().sort_index())

=== Cluster profiles ===
          recency_days  frequency  total_revenue  return_rate  avg_ticket  \
cluster2                                                                    
0               153.42       2.02         166.25         0.12       83.15   
1               550.00       1.13          89.66         0.01       79.85   
2               187.33       1.17          90.61         0.00       78.62   
3               259.88       1.43         105.61         0.51       75.43   
4               150.65       4.27         401.44         0.15      101.62   

          pct_orders_2017  n_genders  avg_days_between_orders  \
cluster2                                                        
0                    0.52       1.52                   367.46   
1                    0.00       1.08                   114.82   
2                    0.99       1.08                   116.07   
3                    0.64       1.38                   100.14   
4                    0.61       2.28         

In [103]:
# Train one model per cluster
cluster_models2 = {}

for cluster_id in range(5):
    train_cluster = df_train_final[df_train_final["cluster2"] == cluster_id]
    X_c = train_cluster[feature_cols]
    y_c = np.log1p(train_cluster["revenue_2018_2019"])
    
    model = XGBRegressor(
        n_estimators=700,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.7,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
    model.fit(X_c, y_c)
    cluster_models2[cluster_id] = model
    print(f"Cluster {cluster_id} trained — {len(train_cluster)} customers")

# Predict
y_pred_cluster2 = np.zeros(len(df_valid_final))
for cluster_id in range(5):
    mask = df_valid_final["cluster2"] == cluster_id
    if mask.sum() > 0:
        y_pred_cluster2[mask] = np.expm1(
            cluster_models2[cluster_id].predict(X_valid[mask])
        )

y_pred_cluster2 = np.clip(y_pred_cluster2, 0, None)

mae_cluster2 = mean_absolute_error(y_valid, y_pred_cluster2)
spearman_cluster2 = spearmanr(y_valid, y_pred_cluster2).statistic

print(f"\nMAE (clustering v2):      {mae_cluster2:.2f}")
print(f"Spearman (clustering v2): {spearman_cluster2:.3f}")
print(f"\nTarget to beat -> MAE: 61.15 | Spearman: 0.41")

Cluster 0 trained — 6203 customers
Cluster 1 trained — 27230 customers
Cluster 2 trained — 39693 customers
Cluster 3 trained — 11837 customers
Cluster 4 trained — 8309 customers

MAE (clustering v2):      66.10
Spearman (clustering v2): 0.380

Target to beat -> MAE: 61.15 | Spearman: 0.41
